In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# ==========================================================
# Load Dataset
# ==========================================================

newsgroups = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
)

documents = newsgroups.data

print("Number of documents:", len(documents))

# ==========================================================
# Convert Text to Bag-of-Words
# ==========================================================

vectorizer = CountVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(documents)

vocabulary = vectorizer.get_feature_names_out()

print("Vocabulary Size:", len(vocabulary))

# ==========================================================
# Total Word Counts
# ==========================================================

word_counts = np.asarray(X.sum(axis=0)).flatten()

total_words = word_counts.sum()

print("Total Words:", total_words)

# ==========================================================
# MLE Estimation
# ==========================================================

theta_mle = word_counts / total_words

# ==========================================================
# MAP Estimation Function
# ==========================================================

def map_estimate(counts, alpha):

    K = len(counts)

    numerator = counts + alpha - 1
    denominator = counts.sum() + K * (alpha - 1)

    theta_map = numerator / denominator

    return theta_map

# ==========================================================
# Different Priors
# ==========================================================

alphas = [0.5, 1, 2, 10]

results = {}

for alpha in alphas:

    theta = map_estimate(word_counts, alpha)

    results[alpha] = theta

# ==========================================================
# Display Top Words
# ==========================================================

top = 10

print("\nTop words using MLE\n")

idx = np.argsort(theta_mle)[::-1][:top]

for i in idx:
    print(f"{vocabulary[i]:15s} {theta_mle[i]:.6f}")

# ==========================================================
# Display MAP Results
# ==========================================================

for alpha in alphas:

    print("\n")
    print("="*40)
    print("MAP with alpha =", alpha)
    print("="*40)

    theta = results[alpha]

    idx = np.argsort(theta)[::-1][:top]

    for i in idx:
        print(f"{vocabulary[i]:15s} {theta[i]:.6f}")

# ==========================================================
# Comparison Table
# ==========================================================

comparison = pd.DataFrame({
    "Word": vocabulary[idx],
    "MLE": theta_mle[idx],
    "MAP(alpha=0.5)": results[0.5][idx],
    "MAP(alpha=1)": results[1][idx],
    "MAP(alpha=2)": results[2][idx],
    "MAP(alpha=10)": results[10][idx]
})

print("\nComparison Table\n")
print(comparison)

# ==========================================================
# L1 Distance between MLE and MAP
# ==========================================================

print("\nDifference between MLE and MAP\n")

for alpha in alphas:

    distance = np.sum(np.abs(theta_mle - results[alpha]))

    print(f"alpha={alpha:<4}  L1 Distance = {distance:.6f}")

Number of documents: 11314
Vocabulary Size: 5000
Total Words: 885959

Top words using MLE

ax              0.070417
max             0.005175
people          0.004631
like            0.004474
don             0.004385
just            0.004235
know            0.003936
use             0.003588
think           0.003399
time            0.003350


MAP with alpha = 0.5
ax              0.070616
max             0.005189
people          0.004644
like            0.004486
don             0.004397
just            0.004246
know            0.003946
use             0.003598
think           0.003408
time            0.003359


MAP with alpha = 1
ax              0.070417
max             0.005175
people          0.004631
like            0.004474
don             0.004385
just            0.004235
know            0.003936
use             0.003588
think           0.003399
time            0.003350


MAP with alpha = 2
ax              0.070023
max             0.005147
people          0.004606
like            0.0